### Definitions

In [479]:
import os
import pandas as pd
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np

PATH_EXAMPLE_CV = "./data/cv_example.pdf"
EMBEDDING_MODEL = "gemini-embedding-001"
LLM_MODEL_NAME = "gemini-2.5-flash-lite"
LLM_PROVIDER = "google_genai"
LLM_MODEL_VECTOR_DIMENSIONS = 3072
LLM_TEMPERATURE = 0.5
EMBED_BATCH_SIZE = 100
EMBED_CONCURRENCY = 5
EMBED_MAX_RETRIES = 3

DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = os.getenv('GEMINI_API_KEY')
JOB_POSTINGS_TABLE = "job_postings_general"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
INDUSTRIES_TABLE = "work_industries"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

In [480]:
def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=db_user, password=db_pw
    )
    register_vector(conn)
    return conn

def get_resume(resume_id: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s", (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def filter_job_postings(industries: list) -> pd.DataFrame:
    filter_string = f"WHERE ai_industries::TEXT[] && ARRAY[{industries}]"
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} {filter_string}")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_position_skills(jobs_ids: list, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE job_id = ANY(%s)", (jobs_ids,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_candidate_skills(resume_id: str, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE resume_id = '{resume_id}'")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)    

def cosine_similarities_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    zero_mask = np.all(matrix == 0, axis=-1)  
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=-1)  
    dot_products = matrix @ query
    similarities = dot_products / (row_norms * query_norm + 1e-10)
    similarities[zero_mask] = 0.0
    return similarities  

def find_skill_compliance(
        skill_embedding: np.ndarray, 
        vector_matrix: np.ndarray, 
        threshold: float, 
        weight_matrix: np.ndarray = None, 
        weight: float = 0):
    
    cosine_similarities = cosine_similarities_matrix(skill_embedding, vector_matrix)
    binary_mask = (cosine_similarities > threshold).astype(np.int8)
    weights = []
    
    if weight != 0 and weight_matrix is not None:
        minimum_weights_matrix = np.where(binary_mask == 1, weight_matrix, 0)    
        binary_mask = ((weight >= minimum_weights_matrix) & (binary_mask == 1)).astype(np.int8)

    mappings = list(zip(*np.where(binary_mask != 0)))  
    weights = [weight_matrix[row, col] for row, col in mappings] if weight_matrix is not None else [0] * len(mappings)
    return mappings, weights

def find_job_matches(mappings: list, skills_df: pd.DataFrame):
    positions_index = set([int(pair[0]) for pair in mappings])
    index_to_job_id = skills_df.drop_duplicates("index").set_index("index")["job_id"]
    return [str(index_to_job_id[i]) for i in positions_index]

def get_compliance_mask(
        compliance_type: str,
        cosine_similarities: np.ndarray, 
        threshold: float,
        weight_matrix: np.ndarray = None, 
        weight: float = 0):    
    if compliance_type == "COMPLIANT":
        binary_mask = (cosine_similarities > threshold).astype(np.int8)
    elif compliance_type == "NONCOMPLIANT":
        binary_mask = (cosine_similarities <= threshold).astype(np.int8)    
    elif compliance_type == "IDEAL" and weight != 0 and weight_matrix is not None:
        minimum_weights_matrix = np.where(binary_mask == 1, weight_matrix, 0)    
        binary_mask = ((weight >= minimum_weights_matrix) & (binary_mask == 1)).astype(np.int8)
    return binary_mask

def create_mapping_matrix(count_list: list):
    nrows = len(count_list)
    ncols = max(count_list)
    matrix = np.zeros((nrows, ncols), dtype=np.int8)
    for i, count in enumerate(count_list):
        matrix[i, :count] = 1
    return matrix     

### Get Candidate Info

In [481]:
RESUME_ID = "b6e8165a-b1af-4117-8a29-4a3a5fc95f32"
resume_df = get_resume(RESUME_ID)
candidate_industries = resume_df["industries"][0]

candidate_hard_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_HARD_SKILLS_TABLE)
candidate_soft_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_SOFT_SKILLS_TABLE)

In [482]:
class MarketSkillsMatrix():
    def __init__(self, skill_type: str, matching_jobs_df: pd.DataFrame):
        self.skill_type = skill_type  # hard or soft
        self.string_matrix = None     # table of actual skill descriptions, where the index represents a job position
        self.weight_matrix = None     # table of skills respective weight values
        self.embedding_matrix = None  # table of skills respective embeddings
        self.mapping = []             # table of skill matches count
        self.count_by_index = []
        self.job_id_by_index = []

        self.mount(matching_jobs_df)
        
    def mount(self, matching_jobs_df):
        matching_jobs_ids = matching_jobs_df["id"].to_list()
        table = SOFT_SKILLS_TABLE if self.skill_type == "soft" else HARD_SKILLS_TABLE
        skills_df = get_position_skills(matching_jobs_ids, table).sort_values("job_id")
        skills_df["index"] = skills_df.groupby("job_id").ngroup()

        string_df = skills_df[["index", "skill_description"]]
        embedding_df = skills_df[["index", "embedding"]]
        weight_df = skills_df[["index", "weight"]]

        skills_size = string_df.groupby('index').size().max()

        self.string_matrix = np.array([
            np.pad(group['skill_description'].values, (0, skills_size - len(group)), constant_values=0)
            for _, group in string_df.groupby('index')
        ])
        self.embedding_matrix = np.array([
            np.vstack(list(group['embedding'].values) + [np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)] * (skills_size - len(group)))
            for _, group in embedding_df.groupby('index')
        ], dtype=np.float32)
        self.weight_matrix = np.array([
            np.pad(group['weight'].values, (0, skills_size - len(group)), constant_values=0)
            for _, group in weight_df.groupby('index')
        ], dtype=np.float32)

        self.count_by_index = skills_df['index']\
                                .value_counts()\
                                .sort_index()\
                                .tolist()
        self.job_id_by_index = skills_df[["index", "job_id"]]\
                                .drop_duplicates()\
                                .job_id\
                                .tolist()
        self.mapping = create_mapping_matrix(self.count_by_index)
    
    def combine(self, array: np.array):
        self.mapping += array    
    

In [483]:
### Market Skills Matrixes
matching_jobs_df = filter_job_postings(candidate_industries)
matching_jobs_ids = matching_jobs_df["id"].to_list()

soft_market = MarketSkillsMatrix("soft", matching_jobs_df.copy())
hard_market = MarketSkillsMatrix("hard", matching_jobs_df.copy())

Find candidate's best matches for each soft skill

In [489]:
SOFT_SKILLS_SIMILARITY_THRESHOLD = 0.69
SOFT_SKILLS_WEIGHT_COLUMN_INDEX = 3
SOFT_SKILLS_STRING_COLUMN_INDEX = 4

skills_count = candidate_soft_skills_df.shape[0] - 1

for i in range(0, 3):
    weight = candidate_soft_skills_df.iloc[(i, SOFT_SKILLS_WEIGHT_COLUMN_INDEX)] 
    skill_embedding = candidate_soft_skills_df.iloc[(i, SOFT_SKILLS_STRING_COLUMN_INDEX) ] 
    cosine_similarities = cosine_similarities_matrix(skill_embedding, soft_market.embedding_matrix)
    binary_mask = get_compliance_mask("COMPLIANT", cosine_similarities, SOFT_SKILLS_SIMILARITY_THRESHOLD)
    soft_market.combine(binary_mask)
    
admissible_matches_mask = soft_market.mapping > 1
admissible_matches_count = admissible_matches_mask.sum(axis=1).tolist()
admitance_percentage_by_job = [
    round(100 * a / b, 2)
    for a, b in zip(admissible_matches_count, soft_market.count_by_index)
]
    
noncompliance_matches_mask = soft_market.mapping == 1    
print("Count of required skills by job: ", soft_market.count_by_index)
print("Percentage attended by candidate: ", admitance_percentage_by_job)

Count of required skills by job:  [12, 5, 5, 12, 6, 21, 28, 16, 5, 5, 12]
Percentage attended by candidate:  [33.33, 60.0, 60.0, 41.67, 66.67, 14.29, 14.29, 12.5, 60.0, 40.0, 8.33]


In [ ]:
# for i in range(0, 3):
#     ideal_compliance = find_skill_compliance(skill_embedding, soft_market.embedding_matrix, SOFT_SKILLS_SIMILARITY_THRESHOLD, soft_market.weight_matrix, weight)
#     ideal_jobs_ids = find_job_matches(ideal_compliance[0], market_soft_skills_df)
#     weights = ideal_compliance[1]
#     ideal_matches_count = len(ideal_compliance[0])
#     print(f"Ideal Job IDs ({ideal_matches_count}): {ideal_jobs_ids}\nWeights: {weights}\n")

# mappings == np.argwhere(binary_mask == 0)
# np.argwhere(sum_matrix == 0) # prints positions

Find candidate's best matches for each hard skill

In [488]:
HARD_SKILLS_SIMILARITY_THRESHOLD = 0.69
HARD_SKILLS_STRING_COLUMN_INDEX = 5
HARD_SKILLS_WEIGHT_COLUMN_INDEX = 3
skills_count = candidate_hard_skills_df.shape[0] - 1

for i in range(0, 3):
    weight = candidate_hard_skills_df.iloc[(i, HARD_SKILLS_WEIGHT_COLUMN_INDEX)] 
    skill_embedding = candidate_hard_skills_df.iloc[(i, HARD_SKILLS_STRING_COLUMN_INDEX) ] 
    cosine_similarities = cosine_similarities_matrix(skill_embedding, hard_market.embedding_matrix)
    binary_mask = get_compliance_mask("COMPLIANT", cosine_similarities, HARD_SKILLS_SIMILARITY_THRESHOLD)
    hard_market.combine(binary_mask)
    
admissible_matches_mask = hard_market.mapping > 1
admissible_matches_count = admissible_matches_mask.sum(axis=1).tolist()
admitance_percentage_by_job = [
    round(100 * a / b, 2)
    for a, b in zip(admissible_matches_count, hard_market.count_by_index)
]

noncompliance_matches_mask = hard_market.mapping == 1   
print("Count of required skills by job: ", hard_market.count_by_index)
print("Percentage attended by candidate: ", admitance_percentage_by_job) 

Count of required skills by job:  [42, 26, 15, 114, 17, 25, 45, 11, 25, 14, 15]
Percentage attended by candidate:  [9.52, 11.54, 20.0, 6.14, 5.88, 12.0, 11.11, 0.0, 12.0, 28.57, 0.0]


### Find candidate's missing skills for market 